# 01 — Amazon Reviews 2023 Beauty 数据集探索性分析（EDA）

**数据集**：Amazon Reviews 2023 — Beauty and Personal Care  
**分析目标**：理解用户购买行为分布规律，为后续推荐模型的特征工程和超参选择提供依据。

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path("../data")
INTERIM = DATA_DIR / "interim"
PROCESSED = DATA_DIR / "processed"
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

print("环境初始化完成")

## 1. 数据加载与基本统计

In [ ]:
# 加载清洗后的交互数据
df = pl.read_parquet(INTERIM / "interactions.parquet")
train = pl.read_parquet(PROCESSED / "train.parquet")
valid = pl.read_parquet(PROCESSED / "valid.parquet")
test = pl.read_parquet(PROCESSED / "test.parquet")

print(f"{'='*50}")
print(f"总交互数：{len(df):>12,}")
print(f"总用户数：{df['user_id'].n_unique():>12,}")
print(f"总商品数：{df['item_id'].n_unique():>12,}")
print(f"时间跨度：{df['year'].min()} — {df['year'].max()}")
print(f"{'='*50}")
print(f"\n数据集划分：")
print(f"  训练集：{len(train):>10,} 条 ({len(train)/len(df)*100:.1f}%)")
print(f"  验证集：{len(valid):>10,} 条 ({len(valid)/len(df)*100:.1f}%)")
print(f"  测试集：{len(test):>10,} 条 ({len(test)/len(df)*100:.1f}%)")
print(f"\n字段列表：{df.columns}")

In [ ]:
# 评分分布统计
print("评分分布：")
rating_dist = df.group_by("rating").agg(
    pl.len().alias("count")
).sort("rating")
print(rating_dist)

# 稀疏性计算
n_users = df['user_id'].n_unique()
n_items = df['item_id'].n_unique()
n_interactions = len(df)
sparsity = 1 - n_interactions / (n_users * n_items)
print(f"\n矩阵稀疏度：{sparsity:.4%}")
print(f"平均每用户交互数：{n_interactions/n_users:.2f}")
print(f"平均每商品交互数：{n_interactions/n_items:.2f}")

## 2. 长尾效应分析（Power Law Distribution）
电商数据普遍遵循长尾分布：少量热门商品/活跃用户贡献了大量交互。

In [ ]:
user_counts = df.group_by("user_id").agg(pl.len().alias("interaction_count")).sort("interaction_count", descending=True)
user_counts_pd = user_counts.to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, len(user_counts_pd)+1)),
    y=user_counts_pd["interaction_count"],
    mode="lines",
    line=dict(color="#2196F3", width=1.5),
    name="用户交互次数"
))
fig.update_layout(
    title="用户活跃度长尾分布（Power Law）",
    xaxis_title="用户排名（按交互次数降序）",
    yaxis_title="交互次数",
    xaxis_type="log",
    yaxis_type="log",
    template="plotly_white",
    width=900, height=450
)
fig.write_image(str(FIGURES / "01_user_longtail.png"), scale=2)
fig.show()

top1_pct = user_counts_pd.head(int(len(user_counts_pd)*0.01))["interaction_count"].sum() / n_interactions * 100
print(f"Top 1% 用户贡献了 {top1_pct:.1f}% 的交互量")

In [ ]:
item_counts = df.group_by("item_id").agg(pl.len().alias("review_count")).sort("review_count", descending=True)
item_counts_pd = item_counts.to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, len(item_counts_pd)+1)),
    y=item_counts_pd["review_count"],
    mode="lines",
    line=dict(color="#F44336", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(244,67,54,0.1)",
    name="商品评论数"
))
fig.update_layout(
    title="商品热度长尾分布（Power Law）",
    xaxis_title="商品排名（按评论数降序）",
    yaxis_title="评论数",
    xaxis_type="log",
    yaxis_type="log",
    template="plotly_white",
    width=900, height=450
)
fig.write_image(str(FIGURES / "02_item_longtail.png"), scale=2)
fig.show()

top_item_pct = item_counts_pd.head(int(len(item_counts_pd)*0.1))["review_count"].sum() / n_interactions * 100
print(f"Top 10% 商品贡献了 {top_item_pct:.1f}% 的评论量")

## 3. 评分分布分析

In [ ]:
rating_pd = df.group_by("rating").agg(pl.len().alias("count")).sort("rating").to_pandas()
rating_pd["percentage"] = rating_pd["count"] / rating_pd["count"].sum() * 100

colors = ["#EF5350", "#FF8A65", "#FFF176", "#81C784", "#42A5F5"]
fig = go.Figure(data=[
    go.Bar(
        x=rating_pd["rating"].astype(str),
        y=rating_pd["percentage"],
        marker_color=colors,
        text=[f"{p:.1f}%" for p in rating_pd["percentage"]],
        textposition="outside",
    )
])
fig.update_layout(
    title="评分分布（Amazon Beauty 数据集存在明显正向偏差）",
    xaxis_title="评分",
    yaxis_title="占比 (%)",
    template="plotly_white",
    width=700, height=450,
    showlegend=False
)
fig.write_image(str(FIGURES / "03_rating_distribution.png"), scale=2)
fig.show()
print(f"平均评分：{df['rating'].mean():.3f}")
print(f"评分中位数：{df['rating'].median():.1f}")

## 4. 时间趋势分析

In [ ]:
monthly = df.group_by(["year", "month"]).agg(
    pl.len().alias("review_count"),
    pl.col("user_id").n_unique().alias("active_users"),
    pl.col("rating").mean().alias("avg_rating")
).sort(["year", "month"])
monthly_pd = monthly.to_pandas()
monthly_pd["date"] = pd.to_datetime(
    monthly_pd["year"].astype(str) + "-" + monthly_pd["month"].astype(str).str.zfill(2) + "-01"
)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("月度评论量趋势", "月度活跃用户数"))
fig.add_trace(go.Scatter(x=monthly_pd["date"], y=monthly_pd["review_count"],
                          mode="lines+markers", name="评论数", line=dict(color="#2196F3")), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly_pd["date"], y=monthly_pd["active_users"],
                          mode="lines+markers", name="活跃用户", line=dict(color="#4CAF50")), row=2, col=1)
fig.update_layout(template="plotly_white", height=600, width=1000,
                  title_text="月度时间趋势（评论量 & 活跃用户数）")
fig.write_image(str(FIGURES / "04_monthly_trend.png"), scale=2)
fig.show()

In [ ]:
weekday_labels = ["周一","周二","周三","周四","周五","周六","周日"]
weekday_dist = df.group_by("weekday").agg(pl.len().alias("count")).sort("weekday").to_pandas()
weekday_dist["day_name"] = [weekday_labels[i] for i in weekday_dist["weekday"]]

fig = px.bar(weekday_dist, x="day_name", y="count",
             title="星期维度购物行为分布",
             labels={"count": "评论数", "day_name": ""},
             color="count",
             color_continuous_scale="Blues",
             template="plotly_white")
fig.write_image(str(FIGURES / "05_weekday_distribution.png"), scale=2)
fig.show()

## 5. 商品类目分析

In [ ]:
if "category" in df.columns:
    category_dist = (
        df.filter(pl.col("category").is_not_null() & (pl.col("category") != ""))
        .group_by("category")
        .agg(
            pl.len().alias("review_count"),
            pl.col("item_id").n_unique().alias("item_count")
        )
        .sort("review_count", descending=True)
        .head(20)
        .to_pandas()
    )
    fig = px.bar(category_dist, x="review_count", y="category", orientation="h",
                 title="Top 20 商品类目（按评论数）",
                 labels={"review_count": "评论数", "category": "类目"},
                 color="review_count", color_continuous_scale="Viridis",
                 template="plotly_white", height=600)
    fig.update_layout(yaxis=dict(categoryorder="total ascending"))
    fig.write_image(str(FIGURES / "06_top_categories.png"), scale=2)
    fig.show()
else:
    print("类目字段不存在，跳过类目分析")

## 6. 用户行为模式分析

In [ ]:
user_stats = df.group_by("user_id").agg([
    pl.len().alias("interaction_count"),
    pl.col("rating").mean().alias("avg_rating"),
    (pl.col("timestamp_sec").max() - pl.col("timestamp_sec").min()).alias("active_days_sec")
]).to_pandas()
user_stats["active_days"] = user_stats["active_days_sec"] / 86400

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("用户交互次数分布", "用户活跃天数分布"))
fig.add_trace(go.Histogram(x=user_stats["interaction_count"].clip(upper=50),
                            nbinsx=50, name="交互次数",
                            marker_color="#2196F3", opacity=0.8), row=1, col=1)
fig.add_trace(go.Histogram(x=user_stats["active_days"].clip(upper=365),
                            nbinsx=50, name="活跃天数",
                            marker_color="#4CAF50", opacity=0.8), row=1, col=2)
fig.update_layout(template="plotly_white", height=450, width=1000,
                  title_text="用户行为分布（截断至合理范围）", showlegend=False)
fig.write_image(str(FIGURES / "07_user_behavior_distribution.png"), scale=2)
fig.show()

print(f"用户交互次数统计：\n{user_stats['interaction_count'].describe()}")

## 7. 数据集划分验证

确认 train/valid/test 在时间维度上**严格不重叠**（防止数据穿越）。

In [ ]:
# 验证时间不重叠
train_max = train["timestamp_sec"].max()
valid_min = valid["timestamp_sec"].min()
valid_max = valid["timestamp_sec"].max()
test_min = test["timestamp_sec"].min()

print(f"训练集时间范围：{train['timestamp_sec'].min()} — {train_max}")
print(f"验证集时间范围：{valid_min} — {valid_max}")
print(f"测试集时间范围：{test_min} — {test['timestamp_sec'].max()}")
print()
if train_max <= valid_min and valid_max <= test_min:
    print("✅ 时间划分验证通过：train < valid < test，无数据穿越")
else:
    print("⚠️ 时间划分存在重叠，请检查 split.py 实现")

# 可视化时间分布
datasets = ["训练集", "验证集", "测试集"]
sizes = [len(train), len(valid), len(test)]
fig = px.pie(values=sizes, names=datasets,
             title=f"数据集划分比例（总计 {sum(sizes):,} 条）",
             color_discrete_sequence=["#2196F3", "#FF9800", "#4CAF50"],
             template="plotly_white")
fig.write_image(str(FIGURES / "08_dataset_split.png"), scale=2)
fig.show()

## 8. 关键发现总结

| 发现 | 数值/描述 | 影响 |
|------|-----------|------|
| 用户长尾 | Top 1% 用户贡献 ~X% 交互 | 冷启动问题显著，需 K-core 过滤 |
| 商品长尾 | Top 10% 商品贡献 ~Y% 评论 | 热门召回（Top-Pop）有较高上限 |
| 评分偏正 | 均值 ~4.2，众数=5 | 正样本定义不宜用评分阈值，改用交互即为正 |
| 稀疏度 | 矩阵稀疏度 > 99.9% | CF 方法需 Embedding 降维，BPR/ALS 适合 |
| 时间分布 | 存在明显季节性波动 | 时间特征（月份、星期）有效，值得引入排序模型 |

> 详见 [reports/01_数据分析报告.md](../reports/01_数据分析报告.md)